In [1]:
import pandas as pd
from pathlib import Path

DF_PATH = Path("../data/raw/online_retail_II.xlsx")
df = pd.read_excel(DF_PATH)

In [2]:
# look at the dataframe
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# info
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[us]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 102.3 MB


In [4]:
# df shape
df.shape

(525461, 8)

In [5]:
# check duplicate values
df.duplicated().sum()

np.int64(6865)

In [6]:
# drop duplicate values
df = df.drop_duplicates(keep="first")

In [7]:
# NaN values
df.isna().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107833
Country             0
dtype: int64

For this analysis, I am trying to cluster customers by purchasing behaviour, therefore I won't need returns and other types of transactions (for example, three rows in the dataset relate to bad debt).

In [8]:
# drop returns
df = df[df.Quantity > 0]

In [9]:
# drop negative prices (3 rows relate to bad debt)
df = df[df.Price > 0]

In [10]:
# fill Description NaN values with Unknown
df.Description = df.Description.fillna("Unknown")

In [11]:
# change Country to category to optimize performance
df.Country = df.Country.astype("category")

In [12]:
# check object columns
for col in df.select_dtypes(include="object"):
    print(f"\n{col}")
    print(df[col].map(type).value_counts())


Invoice
Invoice
<class 'int'>    504730
<class 'str'>         1
Name: count, dtype: int64

StockCode
StockCode
<class 'int'>    428824
<class 'str'>     75907
Name: count, dtype: int64

Description
Description
<class 'str'>    504731
Name: count, dtype: int64


In [13]:
# change Invoice, StockCode and Description to string for exporting
df.Invoice = df.Invoice.astype("string")
df.StockCode = df.StockCode.astype("string")
df.Description = df.Description.astype("string")

In [14]:
# check min. and max. values before converting numerical values
print(f"Quantity Min.: {df.Quantity.min()} | Max. {df.Quantity.max()}")
print(f"Price Min.: {df.Price.min()} | Max. {df.Price.max()}")

Quantity Min.: 1 | Max. 19152
Price Min.: 0.001 | Max. 25111.09


In [15]:
# change numerical data types for better performance
df = df.astype({"Quantity": "int16", "Price": "float32", "Customer ID": "Int64"})

In [16]:
# check reasonable values for numerical columns
df.select_dtypes(include="number").describe()

,Quantity,Price,Customer ID
count,504731.000000,504731.000000,400916.0
mean,11.516923,4.274692,15361.544074
std,87.337497,64.093330,1680.635823
min,1.000000,0.001000,12346.0
25%,1.000000,1.250000,13985.0
50%,3.000000,2.100000,15311.0
75%,12.000000,4.210000,16805.0
max,19152.000000,25111.089844,18287.0


In [17]:
# inspect unusual categories in Country
df.Country.value_counts()

Country
United Kingdom          466603
EIRE                      9450
Germany                   7645
France                    5514
Netherlands               2728
Spain                     1228
Switzerland               1170
Portugal                  1058
Belgium                   1036
Sweden                     886
Channel Islands            821
Italy                      708
Australia                  630
Cyprus                     533
Austria                    524
Greece                     512
Denmark                    418
United Arab Emirates       399
Norway                     365
Finland                    347
Unspecified                306
USA                        230
Poland                     182
Malta                      170
Japan                      164
Lithuania                  154
Singapore                  117
RSA                        110
Bahrain                    106
Canada                      77
Thailand                    76
Hong Kong                   74


In [18]:
# check the shape now
df.shape

(504731, 8)

In [19]:
# look at performance now after changes
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
Index: 504731 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      504731 non-null  string        
 1   StockCode    504731 non-null  string        
 2   Description  504731 non-null  string        
 3   Quantity     504731 non-null  int16         
 4   InvoiceDate  504731 non-null  datetime64[us]
 5   Price        504731 non-null  float32       
 6   Customer ID  400916 non-null  Int64         
 7   Country      504731 non-null  category      
dtypes: Int64(1), category(1), datetime64[us](1), float32(1), int16(1), string(3)
memory usage: 45.2 MB


In [20]:
# look at the final dataframe
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [21]:
# save the dataframe
df.to_parquet("../data/processed/online_retail_cleaned.parquet", index=False)